# Data Quality and Model Exploration

This notebook checks the structure, completeness, and relationships of the Data Jobs dataset before analysis.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('..') / 'data'
DATA_FILE = DATA_DIR / 'job_postings_flat.csv'
job_postings = pd.read_csv(DATA_FILE, low_memory=False)
job_postings['job_posted_date'] = pd.to_datetime(job_postings['job_posted_date'], errors='coerce')

print(f'Loaded {len(job_postings):,} dashboard-aligned job postings')

Loaded 478,895 dashboard-aligned job postings


In [2]:
table_summary = pd.DataFrame({
    'table': ['job_postings_flat'],
    'rows': [len(job_postings)],
    'columns': [job_postings.shape[1]]
})
table_summary

,table,rows,columns
0,job_postings_flat,478895,17


## Schema and Missing Values

In [3]:
print('Job posting columns and types:')
job_postings.info()

missing_values = (
    job_postings.isna()
    .sum()
    .sort_values(ascending=False)
    .rename('missing_values')
)
missing_values[missing_values > 0].to_frame()

Job posting columns and types:
<class 'pandas.DataFrame'>
RangeIndex: 478895 entries, 0 to 478894
Data columns (total 17 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   job_title_short        478895 non-null  str           
 1   job_title              478894 non-null  str           
 2   job_location           477522 non-null  str           
 3   job_via                478889 non-null  str           
 4   job_schedule_type      470983 non-null  str           
 5   job_work_from_home     478895 non-null  bool          
 6   search_location        478895 non-null  str           
 7   job_posted_date        478895 non-null  datetime64[us]
 8   job_no_degree_mention  478895 non-null  bool          
 9   job_health_insurance   478895 non-null  bool          
 10  job_country            478574 non-null  str           
 11  salary_rate            30411 non-null   str           
 12  salary_year_avg        2

,missing_values
salary_hour_avg,468819
salary_year_avg,458560
salary_rate,448484
job_type_skills,68870
job_skills,68870
job_schedule_type,7912
job_location,1373
job_country,321
company_name,7
job_via,6


In [4]:
key_checks = {
    'duplicate_rows': job_postings.duplicated().sum(),
    'duplicate_job_titles_and_dates': job_postings.duplicated(['job_title', 'job_posted_date']).sum(),
    'missing_job_titles': job_postings['job_title'].isna().sum(),
    'missing_posting_dates': job_postings['job_posted_date'].isna().sum(),
}
pd.Series(key_checks, name='count').to_frame()

,count
duplicate_rows,173
duplicate_job_titles_and_dates,7170
missing_job_titles,1
missing_posting_dates,0


## Flat-File Quality Checks

These checks validate the single dashboard source rather than the older normalized SQL tables.

In [5]:
required_columns = {
    'job_title_short', 'job_posted_date', 'job_work_from_home',
    'salary_year_avg', 'salary_hour_avg', 'company_name', 'job_skills'
}
relationship_checks = {
    'missing_required_columns': len(required_columns - set(job_postings.columns)),
    'rows_without_company_name': job_postings['company_name'].isna().sum(),
    'rows_without_skills': job_postings['job_skills'].isna().sum(),
}
pd.Series(relationship_checks, name='count').to_frame()

,count
missing_required_columns,0
rows_without_company_name,7
rows_without_skills,68870


## Date Coverage

The date range establishes the period represented by this analytical snapshot.

In [ ]:
job_postings['job_posted_date'] = pd.to_datetime(job_postings['job_posted_date'], errors='coerce')
date_summary = job_postings['job_posted_date'].agg(['min', 'max', 'count']).to_frame().T
date_summary

,min,max,count
job_posted_date,2024-01-01 00:00:01,2024-12-31 23:37:38,478895


: 